### Class Weights + Random Oversampling

[x] Logistic regression

[x] SVM

[x] Random Forests (for baselining)

[x] XGBoost

[x] K-NN

[x] Naive Bayes

In [1]:
import sys
from pathlib import Path

# cwd can be repo root, src/, or src/feature_selection/ — walk up until src/stroke_data.py exists
_here = Path().resolve()
REPO_ROOT = _here
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "src" / "stroke_data.py").is_file():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise FileNotFoundError("Could not find src/stroke_data.py (open this project from the repo folder).")

sys.path.insert(0, str(REPO_ROOT / "src"))

In [2]:
import sklearn
import scipy
import numpy as np
from stroke_data import get_stroke_data_for_cv, get_stroke_data

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [3]:
X_train, X_test, y_train, y_test = get_stroke_data_for_cv("data/knn-standardize-distance.csv")

In [4]:
from imblearn.over_sampling import RandomOverSampler
from weighted_baselines import run_weighted_baselines
from pprint import pprint

# do random oversampling 
ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

# get the result from running all baselines with the balanced dataset
result = run_weighted_baselines(X_train_res, X_test, y_train_res, y_test)

pprint(result)

{'knn': {'test': {'accuracy': 0.9090019569471625,
                  'f1': 0.041237113402061855,
                  'precision': 0.0425531914893617,
                  'recall': 0.04},
         'train': {'accuracy': 1.0,
                   'f1': 1.0,
                   'precision': 1.0,
                   'recall': 1.0}},
 'lr': {'test': {'accuracy': 0.6771037181996086,
                 'f1': 0.1951219512195122,
                 'precision': 0.1111111111111111,
                 'recall': 0.8},
        'train': {'accuracy': 0.7631781949087169,
                  'f1': 0.7873961218836565,
                  'precision': 0.7143455497382198,
                  'recall': 0.8770892260221137}},
 'nb': {'test': {'accuracy': 0.723091976516634,
                 'f1': 0.2116991643454039,
                 'precision': 0.12297734627831715,
                 'recall': 0.76},
        'train': {'accuracy': 0.7558498328619182,
                  'f1': 0.7655266082232375,
                  'precision': 0.736342

In [5]:
X_train = X_train_res
y_train = y_train_res

In [6]:
y_train.shape

(7778,)

#### Logistic Regression

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

In [7]:
param_lr = {"C": [0.001, 0.01, 0.1, 1.0, 10.0, 100, 1000],
            "solver": ["liblinear", "newton-cg", "newton-cholesky", "sag", "saga"],
            "class_weight": ["balanced"]
            }

lr = LogisticRegression(random_state=42)
grid_search_lr = GridSearchCV(estimator=lr, param_grid=param_lr, scoring="f1") # , verbose=5)
grid_search_lr.fit(X=X_train, y=y_train)

best_lr_model = grid_search_lr.best_estimator_
print("Best params = ", grid_search_lr.best_params_)

best_lr_model.fit(X=X_train, y=y_train)

lr_preds_train = best_lr_model.predict(X_train)
lr_preds = best_lr_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, lr_preds_train))
print("F1 = ", f1_score(y_train, lr_preds_train))
print("Precision = ", precision_score(y_train, lr_preds_train))
print("Recall = ", recall_score(y_train, lr_preds_train))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, lr_preds))
print("F1 = ", f1_score(y_test, lr_preds))
print("Precision = ", precision_score(y_test, lr_preds))
print("Recall = ", recall_score(y_test, lr_preds))

Best params =  {'C': 0.001, 'class_weight': 'balanced', 'solver': 'liblinear'}
-- Train --
Accuracy =  0.7631781949087169
F1 =  0.7873961218836565
Precision =  0.7143455497382198
Recall =  0.8770892260221137
-- Test --
Accuracy =  0.6771037181996086
F1 =  0.1951219512195122
Precision =  0.1111111111111111
Recall =  0.8


### SVM 

https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html

In [8]:
param_svm = {"C": [0.001, 0.01, 0.1, 1.0, 10.0],
             "kernel": ["linear", "rbf"], 
             "gamma": [0.01, 0.1, 1, 10, 100, "auto", "scale"],
             "class_weight": ["balanced"]
             }

svm = SVC()
grid_search_svm = GridSearchCV(estimator=svm, param_grid=param_svm, scoring="f1") # , verbose=5)
grid_search_svm.fit(X=X_train, y=y_train)

best_svm_model = grid_search_svm.best_estimator_
print("Best params = ", grid_search_svm.best_params_)

best_svm_model.fit(X=X_train, y=y_train)

svm_preds_train = best_svm_model.predict(X_train)
svm_preds = best_svm_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, svm_preds_train))
print("F1 = ", f1_score(y_train, svm_preds_train))
print("Precision = ", precision_score(y_train, svm_preds_train))
print("Recall = ", recall_score(y_train, svm_preds_train))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, svm_preds))
print("F1 = ", f1_score(y_test, svm_preds))
print("Precision = ", precision_score(y_test, svm_preds))
print("Recall = ", recall_score(y_test, svm_preds))

Best params =  {'C': 1.0, 'class_weight': 'balanced', 'gamma': 100, 'kernel': 'rbf'}
-- Train --
Accuracy =  1.0
F1 =  1.0
Precision =  1.0
Recall =  1.0
-- Test --
Accuracy =  0.9510763209393346
F1 =  0.0
Precision =  0.0
Recall =  0.0


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Random Forests 
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

In [9]:
param_rf = {"n_estimators": [5, 10, 15, 20],
            "max_features": ["sqrt", "log2", None],
            "max_depth": [5, 10, 15, None],
            "max_leaf_nodes": [5, 10, 15, None],
            "bootstrap": [True, False],
            "class_weight": ["balanced_subsample"]
            }

rf = RandomForestClassifier()
grid_search_rf = GridSearchCV(estimator=rf, param_grid=param_rf, scoring="f1") # , verbose=5)
grid_search_rf.fit(X=X_train, y=y_train)

best_rf_model = grid_search_rf.best_estimator_
print("Best params = ", grid_search_rf.best_params_)

best_rf_model.fit(X=X_train, y=y_train)

rf_preds_train = best_rf_model.predict(X_train)
rf_preds = best_rf_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, rf_preds_train))
print("F1 = ", f1_score(y_train, rf_preds_train))
print("Precision = ", precision_score(y_train, rf_preds_train))
print("Recall = ", recall_score(y_train, rf_preds_train))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, rf_preds))
print("F1 = ", f1_score(y_test, rf_preds))
print("Precision = ", precision_score(y_test, rf_preds))
print("Recall = ", recall_score(y_test, rf_preds))

Best params =  {'bootstrap': False, 'class_weight': 'balanced_subsample', 'max_depth': None, 'max_features': 'log2', 'max_leaf_nodes': None, 'n_estimators': 10}
-- Train --
Accuracy =  1.0
F1 =  1.0
Precision =  1.0
Recall =  1.0
-- Test --
Accuracy =  0.9432485322896281
F1 =  0.0
Precision =  0.0
Recall =  0.0


### XGBoost

https://xgboost.readthedocs.io/en/latest/parameter.html

https://xgboost.readthedocs.io/en/latest/python/sklearn_estimator.html

In [10]:
param_xgb = {
            "max_depth": [6, 10, 15, 20],
            "subsample": [0.1, 0.5, 1], # subsampling helps prevent overfitting. High subsampling number=high overfitting change
            "lambda": [0.5],
            "gamma": [0.5, 1 ,2],
            "objective": ["binary:logistic"],
            "eta": [0.1, 0.3, 1],
            "subsample": [0.1],
            "scale_pos_weight": [19]
            }

xgb = XGBClassifier()
grid_search_xgb = GridSearchCV(estimator=xgb, param_grid=param_xgb, scoring="f1")#, verbose=3)
grid_search_xgb.fit(X=X_train, y=y_train)

best_xgb_model = grid_search_xgb.best_estimator_
print("Best params = ", grid_search_xgb.best_params_)

best_xgb_model.fit(X=X_train, y=y_train)


train_xgb_preds = best_xgb_model.predict(X_train)
xgb_preds = best_xgb_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, train_xgb_preds))
print("F1 = ", f1_score(y_train, train_xgb_preds))
print("Precision = ", precision_score(y_train, train_xgb_preds))
print("Recall = ", recall_score(y_train, train_xgb_preds))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, xgb_preds))
print("F1 = ", f1_score(y_test, xgb_preds))
print("Precision = ", precision_score(y_test, xgb_preds))
print("Recall = ", recall_score(y_test, xgb_preds))

Best params =  {'eta': 0.3, 'gamma': 0.5, 'lambda': 0.5, 'max_depth': 15, 'objective': 'binary:logistic', 'scale_pos_weight': 19, 'subsample': 0.1}
-- Train --
Accuracy =  0.978786320390846
F1 =  0.979226992320282
Precision =  0.9592994573260977
Recall =  1.0
-- Test --
Accuracy =  0.8913894324853229
F1 =  0.22377622377622378
Precision =  0.17204301075268819
Recall =  0.32


### Naive Bayes

https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html#sklearn.naive_bayes.GaussianNB

In [11]:
param_nb = {
            "var_smoothing": [1e-11, 1e-10, 1e-9, 1e-8, 1e-7],   # default is 1e-9
            "priors": [[0.5, 0.5]]
            }

nb = GaussianNB()
grid_search_nb = GridSearchCV(estimator=nb, param_grid=param_nb, scoring="f1")#, verbose=3)
grid_search_nb.fit(X=X_train, y=y_train)

best_nb_model = grid_search_nb.best_estimator_
print("Best params = ", grid_search_nb.best_params_)

best_nb_model.fit(X=X_train, y=y_train)

train_nb_preds = best_nb_model.predict(X_train)
nb_preds = best_nb_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, train_nb_preds))
print("F1 = ", f1_score(y_train, train_nb_preds))
print("Precision = ", precision_score(y_train, train_nb_preds))
print("Recall = ", recall_score(y_train, train_nb_preds))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, nb_preds))
print("F1 = ", f1_score(y_test, nb_preds))
print("Precision = ", precision_score(y_test, nb_preds))
print("Recall = ", recall_score(y_test, nb_preds))

Best params =  {'priors': [0.5, 0.5], 'var_smoothing': 1e-11}
-- Train --
Accuracy =  0.7558498328619182
F1 =  0.7655266082232375
Precision =  0.7363420427553444
Recall =  0.7971200822833633
-- Test --
Accuracy =  0.723091976516634
F1 =  0.2116991643454039
Precision =  0.12297734627831715
Recall =  0.76


### KNN

https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html

In [12]:
param_knn = {
    "n_neighbors": [1, 2, 3, 5, 10],
    "weights": ["uniform"]
}

knn = KNeighborsClassifier()
grid_search_knn = GridSearchCV(estimator=knn, param_grid=param_knn, scoring="f1") #, verbose=5)
grid_search_knn.fit(X=X_train, y=y_train)

best_knn_model = grid_search_knn.best_estimator_
print("Best params = ", grid_search_knn.best_params_)

best_knn_model.fit(X=X_train, y=y_train)

train_knn_preds = best_knn_model.predict(X_train)
knn_preds = best_knn_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, train_knn_preds))
print("F1 = ", f1_score(y_train, train_knn_preds))
print("Precision = ", precision_score(y_train, train_knn_preds))
print("Recall = ", recall_score(y_train, train_knn_preds))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, knn_preds))
print("F1 = ", f1_score(y_test, knn_preds))
print("Precision = ", precision_score(y_test, knn_preds))
print("Recall = ", recall_score(y_test, knn_preds))

Best params =  {'n_neighbors': 1, 'weights': 'uniform'}
-- Train --
Accuracy =  1.0
F1 =  1.0
Precision =  1.0
Recall =  1.0
-- Test --
Accuracy =  0.9090019569471625
F1 =  0.041237113402061855
Precision =  0.0425531914893617
Recall =  0.04


In [14]:
best_knn_model.get_params()

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 1,
 'p': 2,
 'weights': 'uniform'}